In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [4]:
data = pd.read_csv('../datasets/employee.csv')
data.head()

,Education,JoiningYear,City,PaymentTier,Age,Gender,EverBenched,ExperienceInCurrentDomain,LeaveOrNot
0,Bachelors,2017,Bangalore,3,34,Male,No,0,0
1,Bachelors,2013,Pune,1,28,Female,No,3,1
2,Bachelors,2014,New Delhi,3,38,Female,No,2,0
3,Masters,2016,Bangalore,3,27,Male,No,5,1
4,Masters,2017,Pune,3,24,Male,Yes,2,1


- Convert JoiningYear to datetime, create a 'JoiningDate' column (assume Jan 1)
- Convert PaymentTier to category datatype
- Check and handle any missing values

In [5]:
data['joining_date'] = pd.to_datetime(data['JoiningYear'].astype(str) + '-01-01') 
data['payment_tier'] = data['PaymentTier'].map({1: 'gold', 2: 'silver', 3: 'bronze'}).astype('category')

In [6]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4653 entries, 0 to 4652
Data columns (total 11 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   Education                  4653 non-null   object        
 1   JoiningYear                4653 non-null   int64         
 2   City                       4653 non-null   object        
 3   PaymentTier                4653 non-null   int64         
 4   Age                        4653 non-null   int64         
 5   Gender                     4653 non-null   object        
 6   EverBenched                4653 non-null   object        
 7   ExperienceInCurrentDomain  4653 non-null   int64         
 8   LeaveOrNot                 4653 non-null   int64         
 9   joining_date               4653 non-null   datetime64[ns]
 10  payment_tier               4653 non-null   category      
dtypes: category(1), datetime64[ns](1), int64(5), object(4)
memory usage: 

In [7]:
data.isna().sum()

Education                    0
JoiningYear                  0
City                         0
PaymentTier                  0
Age                          0
Gender                       0
EverBenched                  0
ExperienceInCurrentDomain    0
LeaveOrNot                   0
joining_date                 0
payment_tier                 0
dtype: int64

- Create a new column 'TotalExperience' by calculating years from JoiningYear to current year
- Create 'ExperienceLevel' column: 'Junior' (<3 years), 'Mid' (3-7 years), 'Senior' (>7 years)

In [8]:
current_year = pd.Timestamp.now().year
data['total_experience'] = current_year - data['JoiningYear']
data['experience_level'] = np.where(data['total_experience']<5, 'junior', np.where(data['total_experience']<=10, 'mid', 'senior'))

In [9]:
data.head()

,Education,JoiningYear,City,PaymentTier,Age,Gender,EverBenched,ExperienceInCurrentDomain,LeaveOrNot,joining_date,payment_tier,total_experience,experience_level
0,Bachelors,2017,Bangalore,3,34,Male,No,0,0,2017-01-01,bronze,9,mid
1,Bachelors,2013,Pune,1,28,Female,No,3,1,2013-01-01,gold,13,senior
2,Bachelors,2014,New Delhi,3,38,Female,No,2,0,2014-01-01,bronze,12,senior
3,Masters,2016,Bangalore,3,27,Male,No,5,1,2016-01-01,bronze,10,mid
4,Masters,2017,Pune,3,24,Male,Yes,2,1,2017-01-01,bronze,9,mid


- Calculate the average LeaveOrNot rate by:

    (Education level, City, Gender)

- Find which combination has the highest attrition rate

In [10]:
leave_summary = data.groupby(['Education', 'City', 'Gender']).agg(
    leave_rate = ('LeaveOrNot', 'mean'),
    total_employees = ('LeaveOrNot', 'count'),
    left_count = ('LeaveOrNot', 'sum')
)
leave_summary['leave_rate%'] = leave_summary['leave_rate'].apply(lambda x: f"{x:0.2%}")
leave_summary

leave_rate  total_employees  left_count  \
Education City      Gender                                            
Bachelors Bangalore Female    0.265651              591         157   
                    Male      0.229979             1461         336   
          New Delhi Female    0.234637              358          84   
                    Male      0.072626              179          13   
          Pune      Female    0.940329              486         457   
                    Male      0.155894              526          82   
Masters   Bangalore Female    0.703704               54          38   
                    Male      0.700000               70          49   
          New Delhi Female    0.458333              216          99   
                    Male      0.478405              301         144   
          Pune      Female    0.306931              101          31   
                    Male      0.496183              131          65   
PHD       Bangalore Female    0.214286               14           3   
                    Male      0.315789               38          12   
          New Delhi Female    0.260870               46          12   
                    Male      0.245614               57          14   
          Pune      Female    0.333333                9           3   
                    Male      0.066667               15           1   

                           leave_rate%  
Education City      Gender              
Bachelors Bangalore Female      26.57%  
                    Male        23.00%  
          New Delhi Female      23.46%  
                    Male         7.26%  
          Pune      Female      94.03%  
                    Male        15.59%  
Masters   Bangalore Female      70.37%  
                    Male        70.00%  
          New Delhi Female      45.83%  
                    Male        47.84%  
          Pune      Female      30.69%  
                    Male        49.62%  
PHD       Bangalore Female      21.43%  
                    Male        31.58%  
          New Delhi Female      26.09%  
                    Male        24.56%  
          Pune      Female      33.33%  
                    Male         6.67%

In [11]:
leave_summary['leave_rate'].idxmax()

('Bachelors', 'Pune', 'Female')

In [12]:
leave_summary.loc[leave_summary['leave_rate'].idxmax()]

leave_rate         0.940329
total_employees         486
left_count              457
leave_rate%          94.03%
Name: (Bachelors, Pune, Female), dtype: object

- Find all employees who:
- - Have Masters degree
- - Are from Bangalore or Pune
- - Have >3 years experience
- - Are NOT benched

In [13]:
data.head()

,Education,JoiningYear,City,PaymentTier,Age,Gender,EverBenched,ExperienceInCurrentDomain,LeaveOrNot,joining_date,payment_tier,total_experience,experience_level
0,Bachelors,2017,Bangalore,3,34,Male,No,0,0,2017-01-01,bronze,9,mid
1,Bachelors,2013,Pune,1,28,Female,No,3,1,2013-01-01,gold,13,senior
2,Bachelors,2014,New Delhi,3,38,Female,No,2,0,2014-01-01,bronze,12,senior
3,Masters,2016,Bangalore,3,27,Male,No,5,1,2016-01-01,bronze,10,mid
4,Masters,2017,Pune,3,24,Male,Yes,2,1,2017-01-01,bronze,9,mid


In [14]:
mask_city = (data['City']=='Bangalore') | (data['City']=='Pune')
mask = (data['Education']=='Masters') & (mask_city) & (data['total_experience']>3) & (data['EverBenched']=='No')
data.loc[mask].head()

,Education,JoiningYear,City,PaymentTier,Age,Gender,EverBenched,ExperienceInCurrentDomain,LeaveOrNot,joining_date,payment_tier,total_experience,experience_level
3,Masters,2016,Bangalore,3,27,Male,No,5,1,2016-01-01,bronze,10,mid
10,Masters,2012,Bangalore,3,27,Male,No,5,1,2012-01-01,bronze,14,senior
57,Masters,2014,Pune,3,39,Female,No,2,0,2014-01-01,bronze,12,senior
59,Masters,2017,Pune,2,36,Male,No,2,1,2017-01-01,silver,9,mid
69,Masters,2017,Bangalore,3,40,Female,No,2,1,2017-01-01,bronze,9,mid


- Create a 'RiskScore' column using a function:
- - Start with 0
- - +1 if Age > 35
- - +1 if EverBenched is Yes
- - +1 if ExperienceInCurrentDomain < 3
- - +1 if PaymentTier is 1 or 2
- Categorize as 'Low' (0-1), 'Medium' (2), 'High' (3-4)

In [27]:
def risk_score(row):
    score = 0
    score += (row['Age'] > 35)
    score += (row['EverBenched'] == 'Yes')
    score += (row['ExperienceInCurrentDomain'] < 3)
    score += (row['PaymentTier'] in [1, 2])
    
    return int(score)

# Apply row-wise
data['score'] = data.apply(risk_score, axis=1)
data['score_category'] = np.where(data['score'].isin([0, 1]), 'low', np.where(data['score'] == 2, 'mid', 'high'))
data.head()

,Education,JoiningYear,City,PaymentTier,Age,Gender,EverBenched,ExperienceInCurrentDomain,LeaveOrNot,joining_date,payment_tier,total_experience,experience_level,score,score_category
0,Bachelors,2017,Bangalore,3,34,Male,No,0,0,2017-01-01,bronze,9,mid,1,low
1,Bachelors,2013,Pune,1,28,Female,No,3,1,2013-01-01,gold,13,senior,1,low
2,Bachelors,2014,New Delhi,3,38,Female,No,2,0,2014-01-01,bronze,12,senior,2,mid
3,Masters,2016,Bangalore,3,27,Male,No,5,1,2016-01-01,bronze,10,mid,0,low
4,Masters,2017,Pune,3,24,Male,Yes,2,1,2017-01-01,bronze,9,mid,2,mid


Build a weighted churn prediction score combining multiple features.

Create a **ChurnScore** with these weights:

| Feature | Weight | Notes |
|---------|--------|-------|
| **Age** | 0.30 | Normalized using `MinMaxScaler` |
| **PaymentTier** | 0.25 | **Inverse:** lower tier = higher risk |
| **EverBenched** | 0.25 | Yes = 1, No = 0 |
| **ExperienceInCurrentDomain** | 0.20 | **Inverse:** lower experience = higher risk |

Identify top 3 employees per City with highest churn risk

In [28]:
from sklearn.preprocessing import MinMaxScaler

In [51]:
class ChurnScore:
    def __init__(self, weights=None):
        self.weights = weights or {
            'age': 0.30,
            'payment': 0.25,
            'benched': 0.25,
            'experience': 0.20
        }
        self.scalers = {}
    
    def fit(self, data):
        self.scalers['age'] = MinMaxScaler()
        self.scalers['age'].fit(data[['Age']])
        
        self.scalers['payment'] = MinMaxScaler()
        self.scalers['payment'].fit(data[['PaymentTier']])
        
        self.scalers['experience'] = MinMaxScaler()
        self.scalers['experience'].fit(data[['ExperienceInCurrentDomain']])
        
        return self
    
    def transform(self, data):
        df = data.copy()
        
        age_scaled = self.scalers['age'].transform(df[['Age']]).flatten()
        payment_scaled = self.scalers['payment'].transform(df[['PaymentTier']]).flatten()
        exp_scaled = self.scalers['experience'].transform(df[['ExperienceInCurrentDomain']]).flatten()
        
        df['churn_score'] = (
            self.weights['age'] * age_scaled +
            self.weights['payment'] * (1 - payment_scaled) +
            self.weights['benched'] * (df['EverBenched'] == 'Yes') +
            self.weights['experience'] * (1 - exp_scaled)
        )
        
        return df
    
    def fit_transform(self, data):
        return self.fit(data).transform(data)
 
scorer = ChurnScore()
data['churn_score'] = scorer.fit_transform(data)['churn_score']
churn_labels = ['low', 'mid', 'high']
churn_conditions = [
    data['churn_score'] <= 0.4,
    data['churn_score'] <= 0.65,
    data['churn_score'] > 0.65
]
data['churn_score_catoegry'] = np.select(churn_conditions, churn_labels)
data.head()

,Education,JoiningYear,City,PaymentTier,Age,Gender,EverBenched,ExperienceInCurrentDomain,LeaveOrNot,joining_date,payment_tier,total_experience,experience_level,score,score_category,churn_score,churn_score_catoegry
0,Bachelors,2017,Bangalore,3,34,Male,No,0,0,2017-01-01,bronze,9,mid,1,low,0.389474,low
1,Bachelors,2013,Pune,1,28,Female,No,3,1,2013-01-01,gold,13,senior,1,low,0.459023,mid
2,Bachelors,2014,New Delhi,3,38,Female,No,2,0,2014-01-01,bronze,12,senior,2,mid,0.395489,low
3,Masters,2016,Bangalore,3,27,Male,No,5,1,2016-01-01,bronze,10,mid,0,low,0.136090,low
4,Masters,2017,Pune,3,24,Male,Yes,2,1,2017-01-01,bronze,9,mid,2,mid,0.424436,mid


In [55]:
data['city_rank'] = data.groupby('City')['churn_score'].rank(method='dense', ascending=False)
data[data['city_rank'] <= 3].sort_values(['City', 'city_rank'])

,Education,JoiningYear,City,PaymentTier,Age,Gender,EverBenched,ExperienceInCurrentDomain,LeaveOrNot,joining_date,payment_tier,total_experience,experience_level,score,score_category,churn_score,churn_score_catoegry,CityRank,city_rank
3358,Bachelors,2018,Bangalore,1,36,Male,Yes,1,0,2018-01-01,gold,8,mid,4,high,0.892481,high,1,1.0
4067,Bachelors,2017,Bangalore,1,34,Male,Yes,0,0,2017-01-01,gold,9,mid,3,high,0.889474,high,2,2.0
3869,Bachelors,2015,Bangalore,1,32,Male,Yes,0,1,2015-01-01,gold,11,senior,3,high,0.857895,high,3,3.0
3979,Masters,2017,New Delhi,2,40,Female,Yes,2,0,2017-01-01,silver,9,mid,4,high,0.802068,high,1,1.0
3177,Masters,2017,New Delhi,2,36,Male,Yes,2,0,2017-01-01,silver,9,mid,4,high,0.738910,high,2,2.0
3432,Masters,2017,New Delhi,2,36,Male,Yes,2,0,2017-01-01,silver,9,mid,4,high,0.738910,high,2,2.0
3579,Masters,2017,New Delhi,1,40,Female,No,0,0,2017-01-01,gold,9,mid,3,high,0.734211,high,3,3.0
3643,Masters,2017,New Delhi,1,40,Male,No,0,0,2017-01-01,gold,9,mid,3,high,0.734211,high,3,3.0
4490,Bachelors,2013,Pune,1,39,Female,Yes,0,1,2013-01-01,gold,13,senior,4,high,0.968421,high,1,1.0
4141,Bachelors,2015,Pune,2,40,Female,Yes,0,1,2015-01-01,silver,11,senior,4,high,0.859211,high,2,2.0


Analyze how retention varies by joining year cohorts and education level.

Group employees by **JoiningYear** and **Education**:

For each cohort, calculate:

| Metric | Description |
|--------|-------------|
| **Total Employees Hired** | Count of employees in that cohort |
| **Number Who Left** | Count where `LeaveOrNot == 1` |
| **Retention Rate** | Percentage of employees who stayed |
| **Average Tenure** | Mean `YearsSinceJoining` for those who stayed |


In [77]:
def calucalate_cohort_stats(group):
    total = group.size
    left_count = group['LeaveOrNot'].sum()
    stayed_count = total - left_count

    avg_tenure_stayed = group[group['LeaveOrNot'] == 0]['total_experience'].mean()
    avg_tenure_left = group[group['LeaveOrNot'] == 1]['total_experience'].mean()
    
    avg_tenure_stayed = 0 if pd.isna(avg_tenure_stayed) else avg_tenure_stayed
    avg_tenure_left = 0 if pd.isna(avg_tenure_left) else avg_tenure_left
    
    return pd.Series({
        'total_employees': total,
        'left': left_count,
        'stayed': stayed_count,
        'left_rate': round((left_count * 100 / total), 2) if total > 0 else 0,
        'retention_rate': round((stayed_count * 100 / total), 2) if total > 0 else 0,
        'avg_tenure_all': round(group['total_experience'].mean(), 2) if total > 0 else 0,
        'avg_tenure_stayed': round(avg_tenure_stayed, 2),
        'avg_tenure_left': round(avg_tenure_left, 2),
        'min_tenure': group['total_experience'].min(),
        'max_tenure': group['total_experience'].max(),
        'std_tenure': round(group['total_experience'].std(), 2) if total > 1 else 0
    })

cohort_stats = data.groupby(['JoiningYear', 'Education']).apply(calucalate_cohort_stats, include_groups=False).reset_index()
cohort_stats

,JoiningYear,Education,total_employees,left,stayed,left_rate,retention_rate,avg_tenure_all,avg_tenure_stayed,avg_tenure_left,min_tenure,max_tenure,std_tenure
0,2012,Bachelors,7497.0,82.0,7415.0,1.09,98.91,14.0,14.0,14.0,14.0,14.0,0.0
1,2012,Masters,833.0,27.0,806.0,3.24,96.76,14.0,14.0,14.0,14.0,14.0,0.0
2,2012,PHD,238.0,0.0,238.0,0.00,100.00,14.0,14.0,0.0,14.0,14.0,0.0
3,2013,Bachelors,9027.0,147.0,8880.0,1.63,98.37,13.0,13.0,13.0,13.0,13.0,0.0
4,2013,Masters,1666.0,67.0,1599.0,4.02,95.98,13.0,13.0,13.0,13.0,13.0,0.0
5,2013,PHD,680.0,10.0,670.0,1.47,98.53,13.0,13.0,13.0,13.0,13.0,0.0
6,2014,Bachelors,10285.0,145.0,10140.0,1.41,98.59,12.0,12.0,12.0,12.0,12.0,0.0
7,2014,Masters,1258.0,27.0,1231.0,2.15,97.85,12.0,12.0,12.0,12.0,12.0,0.0
8,2014,PHD,340.0,1.0,339.0,0.29,99.71,12.0,12.0,12.0,12.0,12.0,0.0
9,2015,Bachelors,10880.0,278.0,10602.0,2.56,97.44,11.0,11.0,11.0,11.0,11.0,0.0


In [81]:
pivot_retention = cohort_stats.pivot(
    index='JoiningYear',
    columns='Education',
    values='retention_rate'
)
pivot_retention

Education,Bachelors,Masters,PHD
JoiningYear,,,
2012,98.91,96.76,100.00
2013,98.37,95.98,98.53
2014,98.59,97.85,99.71
2015,97.44,97.93,99.64
2016,98.80,97.20,99.20
2017,98.89,97.62,100.00
2018,94.21,94.20,94.12


In [83]:
pivot_yoy = pivot_retention.pct_change(axis=0) * 100
pivot_yoy

Education,Bachelors,Masters,PHD
JoiningYear,,,
2012,NaN,NaN,NaN
2013,-0.545951,-0.806118,-1.470000
2014,0.223645,1.948323,1.197605
2015,-1.166447,0.081758,-0.070204
2016,1.395731,-0.745430,-0.441590
2017,0.091093,0.432099,0.806452
2018,-4.732531,-3.503380,-5.880000


In [84]:
final_table = pivot_retention.join(pivot_yoy, lsuffix='_rate', rsuffix='_yoy')
final_table

Education,Bachelors_rate,Masters_rate,PHD_rate,Bachelors_yoy,Masters_yoy,PHD_yoy
JoiningYear,,,,,,
2012,98.91,96.76,100.00,NaN,NaN,NaN
2013,98.37,95.98,98.53,-0.545951,-0.806118,-1.470000
2014,98.59,97.85,99.71,0.223645,1.948323,1.197605
2015,97.44,97.93,99.64,-1.166447,0.081758,-0.070204
2016,98.80,97.20,99.20,1.395731,-0.745430,-0.441590
2017,98.89,97.62,100.00,0.091093,0.432099,0.806452
2018,94.21,94.20,94.12,-4.732531,-3.503380,-5.880000


Feature Engineering & Dimensionality Reduction

Create derived features and reduce dimensions for modeling.


Part 1: Create New Features

| New Feature | Derivation |
|-------------|------------|
| **CareerStage** | `'Junior'` (Age < 28), `'Mid'` (28-35), `'Senior'` (> 35) |
| **DomainExpertise** | `'Novice'` (Exp < 2), `'Intermediate'` (2-5), `'Expert'` (> 5) |
| **StabilityScore** | `1 / (YearsSinceJoining + 1)` — normalized |
| **CompensationRisk** | `PaymentTier * (1 if EverBenched == 'Yes' else 0)` |


Part 2: Encode Categorical Variables

- Create dummy variables for all categorical columns


Part 3: Correlation Analysis

1. Find **top 3 features** most correlated with `LeaveOrNot`

2. Create a **reduced DataFrame** with only features having correlation **> 0.1** with the target

3. Output the shape of **original vs reduced** dataset


In [89]:
# Part 1: Create new features
data['career_stage'] = pd.cut(data['Age'], bins=[0, 28, 35, 100], labels=['junior', 'mid', 'senior'])
data['domain_expertise'] = pd.cut(data['ExperienceInCurrentDomain'], bins=[0, 2, 5, 100],labels=['novice', 'intermediate', 'expert'])
data['stability_score'] = 1 / (data['total_experience'] + 1)
data['stability_score'] = (data['stability_score'] - data['stability_score'].min()) / (data['stability_score'].max() - data['stability_score'].min())
data['compensation_risk'] = data['PaymentTier'] * (data['EverBenched'] == 'Yes')

# Part 2: Create dummy variables
data_encoded = pd.get_dummies(data, drop_first=True)

# Part 3: Correlation analysis
correlations = data_encoded.corr()['LeaveOrNot'].sort_values(ascending=False)

print(correlations)

print("\n" + "="*50)
print("Top 3 features correlated with LeaveOrNot:")
print(correlations[1:4]) # skip 1.0

# Reduced DataFrame
high_corr_features = correlations[abs(correlations) > 0.1].index.tolist()
data_reduced = data_encoded[high_corr_features]

print(f"\nOriginal shape: {data_encoded.shape}")
print(f"Reduced shape:  {data_reduced.shape}")

LeaveOrNot                       1.000000
payment_tier_silver              0.266426
stability_score                  0.208455
City_Pune                        0.206264
JoiningYear                      0.181705
joining_date                     0.181702
score                            0.154888
Education_Masters                0.145801
churn_score                      0.125338
churn_score_catoegry_mid         0.085942
score_category_mid               0.085225
EverBenched_Yes                  0.078438
compensation_risk                0.058461
payment_tier_gold                0.011065
career_stage_mid                -0.005141
domain_expertise_expert         -0.006342
domain_expertise_intermediate   -0.020476
ExperienceInCurrentDomain       -0.030504
City_New Delhi                  -0.033341
career_stage_senior             -0.038332
Education_PHD                   -0.038938
Age                             -0.051126
city_rank                       -0.070010
CityRank                        -0